# prep.motus

MOTUS is a marker-based taxonomic profiling tool. We used MOTUS to detect bacteria taxa in our libraries, obtaining a set of different MOTUS outputs. In this notebook, we process that data to make it available as a table that contains in each row:

- library
- taxid (NCBI)
- scientific_name
- gtdb genome representative (useful for phylogenetic queries)
- is_pab (whether the bacteria is known to be associated to plants)
- pab_type (the role in the associatiion with plants, if known)

In [16]:
import pandas as pd
import seaborn as sns 
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

┌─────────┬─────────────┐
│  name   │ description │
│ varchar │   varchar   │
├─────────┴─────────────┤
│        0 rows         │
└───────────────────────┘

## Loading MOTUS data

In this case, we are reading raw outputs from MOTUS where I removed most of the unrelevant columns. We read a source file, and then each of the files associated to the source file. 

In [17]:
index_df = pd.read_csv("input/motus-g1/source.csv", sep=",", header=None, index_col=0, names=['library', 'read1', 'read2'])
index_df

,library,read1,read2
1,PV001,PV1_18807_ATCACG_read1.fastq.gz,PV1_18807_ATCACG_read2.fastq.gz
2,PV002,PV2_18808_TGACCA_read1.fastq.gz,PV2_18808_TGACCA_read2.fastq.gz
3,PV003,PV3_16716_GGCTAC_read1.fastq.gz,PV3_16716_GGCTAC_read2.fastq.gz
4,PV003bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAACRAAPEI-203_2.fq.zip
5,PV004bgi,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_1.fq.zip,FCH5VT5ALXX_L5_HKRDPLAgshTAADRAAPEI-204_2.fq.zip
...,...,...,...
320,PV586,PV586_03661AAB_AGTTGC_read1.fastq.gz,PV586_03661AAB_AGTTGC_read2.fastq.gz
321,PV587,PV587_03662AAB_ACCTTC_read1.fastq.gz,PV587_03662AAB_ACCTTC_read2.fastq.gz
322,PV588,PV588_03663AAB_TCGCAT_read1.fastq.gz,PV588_03663AAB_TCGCAT_read2.fastq.gz
323,PV589,PV589_03664AAB_AGGACT_read1.fastq.gz,PV589_03664AAB_AGGACT_read2.fastq.gz


In [18]:
def load_motus_classification(x, library):
    try:
        u = pd.read_csv(x, sep="\t")
    except pd.errors.EmptyDataError:
        u = pd.DataFrame(data=[], columns=['mOTU', 'taxonomy', 'taxid', 'abundance'])
    u.columns = ['mOTU', 'taxonomy', 'taxid', 'abundance']
    u = u.dropna(subset=['taxid']).drop_duplicates(['taxid'])
    u['library'] = library
    return u

We proceed to read each of the files, and stack them together in a hits dataframe.

In [19]:
hits = []
for _, row in index_df.iterrows():
    u = load_motus_classification("input/motus-g1/{:s}.classification.csv".format(row['library']), row['library'])
    if len(u) > 0:
        for _, item in u.iterrows():
            hits.append(
                {
                    'library': item.library,
                    'taxid': int(item.taxid)
                }
            )
hits = pd.DataFrame.from_records(hits)
hits


,library,taxid
0,PV001,2097
1,PV002,59620
2,PV002,2097
3,PV002,69896
4,PV003,641148
...,...,...
1477,PV587,1886670
1478,PV588,59620
1479,PV588,43675
1480,PV589,332410


## Assigning scientific names

Now we use taxoniq to assign scientific names.

In [20]:
import taxoniq

def fault_tolerant_name(x):
    try:
        return taxoniq.Taxon(x).scientific_name
    except KeyError:
        return ""

In [21]:
bacteria_reference = hits[['taxid']].drop_duplicates('taxid').copy() # type: ignore
bacteria_reference['scientific_name'] = bacteria_reference.taxid.apply(lambda x: fault_tolerant_name(x))
bacteria_reference

,taxid,scientific_name
0,2097,Mycoplasmoides genitalium
1,59620,uncultured Clostridium sp.
3,69896,Candidatus Phytoplasma solani
4,641148,Neisseria sp. oral taxon 014
7,1236179,Janthinobacterium sp. B9-8
...,...,...
1437,1835723,Arthrobacter sp. OY3WO11
1442,1955620,Hymenobacter sp. CRA2
1446,679249,Holzapfeliella floricola
1470,866801,Apilactobacillus ozensis


Some analysis can use phylogenetic distances (e.g. Faith's phylogenetic distance). Therefore, we can use the GTDB metadata to find mapping tree leafs, so future analysis can map hits to the Bac120 phylogenetic tree of the GTDB.

**NOTE** This part is deprecated, now that we are not running any analysis that involves phylogenetics.

## Cross-reference with GTDB reference genomes

In [22]:
# from skbio.tree import TreeNode
# tree = TreeNode.read("input/bac120.tree")
# tree_tip_names = list(tree.subset())
# gtdb_metadata = pd.read_csv("input/bac120_metadata.tsv", sep="\t")
# gtdb_metadata[gtdb_metadata['gtdb_genome_representative'].apply(lambda x: x.replace("_", " ")).isin(tree_tip_names)]
# gtdb_accessions = gtdb_metadata[['ncbi_taxid', 'gtdb_genome_representative']].drop_duplicates('ncbi_taxid', keep='first').copy()
# gtdb_accessions['gtdb_genome_representative'] = gtdb_accessions['gtdb_genome_representative'].apply(lambda x: x.replace("_", " "))
# # gtdb_accessions.to_csv('scratch/semantic-model/gtdb.csv', sep=',')
# bacteria_reference = pd.merge(bacteria_reference, gtdb_accessions, left_on='taxid', right_on='ncbi_taxid', how='left').drop(columns=['ncbi_taxid'])
bacteria_reference

,taxid,scientific_name
0,2097,Mycoplasmoides genitalium
1,59620,uncultured Clostridium sp.
3,69896,Candidatus Phytoplasma solani
4,641148,Neisseria sp. oral taxon 014
7,1236179,Janthinobacterium sp. B9-8
...,...,...
1437,1835723,Arthrobacter sp. OY3WO11
1442,1955620,Hymenobacter sp. CRA2
1446,679249,Holzapfeliella floricola
1470,866801,Apilactobacillus ozensis


## Merge with Sanchis21 to obtain PAB labels

Sanchis et al. (2021) described a set of Plant-Associated Bacteria (PABs). We will load that set and assign tags to those bacteria hits that are present in Sanchis et al 2021.  

In [23]:
sanchis21 = pd.read_csv("input/sanchis21.tableS1.csv", sep="\t").drop_duplicates('TaxId', keep='first')
sanchis21['is_pab'] = True
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'pab_pathogen' if x['PAB-phyto'] == 'P' else "pab_unknown", axis=1)
sanchis21['pab_type'] = sanchis21.apply(lambda x: 'pab_symbiont' if x['PAB-symb'] == 'S' else x['pab_type'], axis=1)
sanchis21 = sanchis21.rename(columns={'TaxId':'taxid'})
sanchis21

,taxid,Biosample,Representative species,PAB-phyto,PAB-symb,is_pab,pab_type
0,178901,SAMN02849420,Acetobacter malorum,NaN,NaN,True,pab_unknown
1,441768,SAMN02603756,Acholeplasma laidlawii PG-8A,NaN,NaN,True,pab_unknown
2,32002,SAMN06198582,Achromobacter denitrificans,NaN,NaN,True,pab_unknown
3,217204,SAMN06174288,Achromobacter insolitus,NaN,NaN,True,pab_unknown
4,72556,SAMN03941583,Achromobacter piechaudii,NaN,NaN,True,pab_unknown
...,...,...,...,...,...,...,...
955,698414,SAMN06765826,Xylella fastidiosa subsp. pauca,P,NaN,True,pab_pathogen
956,1444770,SAMN02570451,Xylella taiwanensis,P,NaN,True,pab_pathogen
957,1735686,SAMN04151686,Xylophilus sp. Leaf220,NaN,NaN,True,pab_unknown
958,1288385,SAMEA1486427,Yersinia pekkanenii,NaN,NaN,True,pab_unknown


We merge `sanchis21` and our `bacteria_reference`

In [24]:
bacteria_reference = pd.merge(bacteria_reference, sanchis21[['taxid', 'is_pab', 'pab_type']], on='taxid', how='left')
bacteria_reference['is_pab'] = bacteria_reference['is_pab'].fillna(False)
bacteria_reference['pab_type'] = bacteria_reference['pab_type'].fillna('')
bacteria_reference

/tmp/ipykernel_4065417/1982941430.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bacteria_reference['is_pab'] = bacteria_reference['is_pab'].fillna(False)


,taxid,scientific_name,is_pab,pab_type
0,2097,Mycoplasmoides genitalium,False,
1,59620,uncultured Clostridium sp.,False,
2,69896,Candidatus Phytoplasma solani,False,
3,641148,Neisseria sp. oral taxon 014,False,
4,1236179,Janthinobacterium sp. B9-8,False,
...,...,...,...,...
518,1835723,Arthrobacter sp. OY3WO11,True,pab_unknown
519,1955620,Hymenobacter sp. CRA2,False,
520,679249,Holzapfeliella floricola,False,
521,866801,Apilactobacillus ozensis,False,


Finally, we merge the `hits` dataframe and the `bacteria_reference`. 

In [25]:
bacteria_hits = pd.merge(
    hits, bacteria_reference, on='taxid' # type: ignore
)
bacteria_hits

,library,taxid,scientific_name,is_pab,pab_type
0,PV001,2097,Mycoplasmoides genitalium,False,
1,PV002,59620,uncultured Clostridium sp.,False,
2,PV002,2097,Mycoplasmoides genitalium,False,
3,PV002,69896,Candidatus Phytoplasma solani,False,
4,PV003,641148,Neisseria sp. oral taxon 014,False,
...,...,...,...,...,...
1477,PV587,1886670,Paenibacillus nuruki,False,
1478,PV588,59620,uncultured Clostridium sp.,False,
1479,PV588,43675,Rothia mucilaginosa,False,
1480,PV589,332410,,False,


## Save

We proceed to save the results

**Note**: This way of saving results is deprecated. It will be removed in the future.

In [26]:
bacteria_hits.to_csv("output/hits.bacteria.csv", sep=";", index=None) # type: ignore

In [27]:
bacteria_hits.query('is_pab == True').to_csv("output/hits.pab-bacteria.csv", sep=";", index=None) # type: ignore

Now, we will save them to the local database.

In [28]:
db.save_dataframe(
    bacteria_hits, table_name="D_bacteriaHits", 
    description="This table contains all the MOTUS hits obtained, regardless of their status. It contains the library where the detection took place, taxid, scientific name and the PAB labels"
)

Saved D_bacteriaHits to db.2025-10-27


In [29]:
db.save_dataframe(
    bacteria_hits.query('is_pab == True'), table_name="D_PABHits", 
    description="This table contains all the PAB MOTUS hits. It contains the library where the detection took place, taxid, scientific name and the PAB labels"
)

Saved D_PABHits to db.2025-10-27


In [30]:
db.conn.close()